# ANÁLISE EXPLORATORIA

In [ ]:
# =====================================================================
# 1. UPLOAD E IMPORTAÇÕES
# =====================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
from google.colab import files
from IPython.display import display # Para mostrar tabelas bonitas no Colab

# Configuração do estilo dos gráficos
sns.set_theme(style="whitegrid")

print("Por favor, faça o upload do arquivo 'dataset_ambiental.csv':")
# Abre a caixa de diálogo para upload no Colab
uploaded = files.upload()

# Pega o nome do arquivo que foi feito upload (esperado: 'dataset_ambiental.csv')
nome_arquivo = list(uploaded.keys())[0]

# Carrega o dataframe
df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]))

print("\n" + "="*50)
print("🔍 INÍCIO DA ANÁLISE EXPLORATÓRIA - CRISP-DM")
print("="*50)

# Definindo a variável alvo globalmente para uso nas seções
target = 'Qualidade_Ambiental'

# =====================================================================
# 2. ESTATÍSTICAS DESCRITIVAS E DISTRIBUIÇÃO DO ALVO
# =====================================================================
print("\n📊 Resumo Estatístico dos Dados (Describe):")
# O .describe() traz média, desvio padrão, min, max e quartis das colunas numéricas
estatisticas = df.describe()
display(estatisticas)

if target in df.columns:
    print(f"\n📊 Distribuição da Variável Alvo ({target}):")
    # Pega a contagem das classes
    contagem_alvo = df[target].value_counts()

    # Exibe a tabela
    display(contagem_alvo.to_frame())

    # Gera o gráfico de barras da distribuição
    plt.figure(figsize=(10, 5))
    # hue atribuído para evitar o warning do palette no seaborn
    sns.barplot(x=contagem_alvo.index, y=contagem_alvo.values, hue=contagem_alvo.index, palette="viridis", legend=False)
    plt.title(f'Gráfico de Barras: Distribuição das Classes - {target}', fontsize=14)
    plt.xlabel('Classes', fontsize=12)
    plt.ylabel('Frequência (Quantidade de Registros)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# =====================================================================
# 3. ANÁLISE DE QUALIDADE: DADOS NULOS E INCONSISTENTES
# =====================================================================
print("\nVerificando dados nulos e inconsistentes na tipagem...")

# Definindo as colunas que deveriam ser puramente numéricas
colunas_numericas_esperadas = ['Temperatura', 'Umidade', 'CO2', 'CO', 'Pressao_Atm', 'NO2', 'SO2', 'O3']

nulos_originais = df.isnull().sum()
inconsistencias = {}

for col in colunas_numericas_esperadas:
    # Tenta converter para numérico; o que for texto/símbolo vira NaN
    convertido = pd.to_numeric(df[col], errors='coerce')

    # A diferença entre os NaNs após conversão e os NaNs originais
    # representa dados que não correspondiam ao formato correto (ex: strings/letras)
    qtd_inconsistentes = convertido.isnull().sum() - nulos_originais[col]
    inconsistencias[col] = qtd_inconsistentes

# Criando um DataFrame de Qualidade
df_qualidade = pd.DataFrame({
    'Valores Nulos': nulos_originais,
    'Valores Inconsistentes (Tipo Errado)': pd.Series(inconsistencias)
}).fillna(0) # Preenche 0 para variáveis não verificadas na conversão (ex: target)

# Plot: Nulos e Inconsistentes
plt.figure(figsize=(12, 6))
df_qualidade[['Valores Nulos', 'Valores Inconsistentes (Tipo Errado)']].plot(kind='bar', stacked=True, colormap='viridis', ax=plt.gca())
plt.title('Qualidade dos Dados: Valores Nulos e Inconsistentes por Coluna', fontsize=14)
plt.ylabel('Quantidade de Registros', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title="Tipo de Problema")
plt.tight_layout()
plt.show()

# =====================================================================
# 4. DISTRIBUIÇÃO DOS DADOS (VARIÁVEIS NUMÉRICAS)
# =====================================================================
print("\nGerando gráficos de distribuição (Histogramas)...")
colunas_numericas = df.select_dtypes(include=[np.number]).columns
n_cols = len(colunas_numericas)

fig, axes = plt.subplots(nrows=(n_cols+2)//3, ncols=3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(colunas_numericas):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Distribuição: {col}')
    axes[i].set_ylabel('Frequência')

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j]) # Remove os subplots vazios

plt.tight_layout()
plt.show()

# =====================================================================
# 5. OUTLIERS (VALORES ATÍPICOS)
# =====================================================================
print("\nGerando gráficos de Boxplot (Outliers)...")
fig, axes = plt.subplots(nrows=(n_cols+2)//3, ncols=3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(colunas_numericas):
    sns.boxplot(y=df[col].dropna(), ax=axes[i], color='lightgreen')
    axes[i].set_title(f'Boxplot: {col}')

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# =====================================================================
# 6. MATRIZ DE CORRELAÇÃO
# =====================================================================
print("\nGerando Matriz de Correlação...")
plt.figure(figsize=(10, 8))
# Calcula a correlação de Pearson ignorando os nulos
corr = df[colunas_numericas].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlação das Variáveis Numéricas', fontsize=15)
plt.tight_layout()
plt.show()

# =====================================================================
# 7. RELAÇÃO E QUALIDADE: FEATURES VS VARIÁVEL ALVO
# =====================================================================
if target in df.columns:
    print(f"\nGerando relação das variáveis numéricas com o alvo ({target})...")
    fig, axes = plt.subplots(nrows=(n_cols+2)//3, ncols=3, figsize=(16, 12))
    axes = axes.flatten()

    # Ordem das classes para manter o padrão visual lógico (da melhor para a pior)
    ordem_classes = ['Excelente', 'Boa', 'Moderada', 'Ruim', 'Muito Ruim']
    # Filtra apenas as classes que existem no dataset para a ordem
    ordem_presente = [c for c in ordem_classes if c in df[target].unique()]

    for i, col in enumerate(colunas_numericas):
        sns.boxplot(x=target, y=col, data=df, ax=axes[i], hue=target, palette="Set2", order=ordem_presente, legend=False)
        axes[i].set_title(f'Relação: {col} vs {target}')
        axes[i].tick_params(axis='x', rotation=45)

    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

else:
    print(f"\nAviso: A coluna '{target}' não foi encontrada no dataset para a análise final.")